# Movies Data Collection 

## Objective

The objective of this notebook is to collect information on movies adaptations from Amazon Prime Video and Netflix and enrich the dataset with IMDb metadata for further analysis.

## Data Sources

* Wikipedia pages containing lists of Netflix/Amazon original titles.
* OMDb API for IMDb metadata such as ratings, vote counts, genres, release years, and content ratings.

## Methodology

1. Titles were collected through web scraping from Wikipedia.
2. IMDb metadata was retrieved using the OMDb API.
3. Each title was manually verified to determine whether it was based on a book.
4. Only confirmed book adaptations were retained for further analysis.


# Amazon Movies Data Collection

## Web Scraping Wikipedia Data

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
url = "https://en.wikipedia.org/wiki/List_of_Amazon_Prime_Video_original_films"

headers = {
    "User-Agent": "Mozilla/5.0"
}

html = requests.get(url, headers=headers).text

In [3]:
from io import StringIO

tables = pd.read_html(StringIO(html))

print(len(tables))

59


In [4]:
for i, table in enumerate(tables):
    print(f"\nTABLE {i}")
    print(table.head())


TABLE 0
                      Title          Genre(s)            Premiere Language
0             Brad's Status             Drama  September 15, 2017  English
1                 Pass Over             Drama      April 20, 2018  English
2                  Peterloo  Historical drama       April 5, 2019  English
3           Guava Island[1]             Drama      April 13, 2019  English
4  Brittany Runs a Marathon            Comedy   November 22, 2019  English

TABLE 1
                                               Title           Premiere  \
0                  Jonas Brothers: Chasing Happiness       June 4, 2019   
1                           Andy Murray: Resurfacing  November 29, 2019   
2                Alejandro Sanz: #Lagira de #eldisco   December 5, 2019   
3  Happiness Continues: A Jonas Brothers Concert ...     April 24, 2020   
4                           Amaia: Una vuelta al sol        May 1, 2020   

  Language  
0  English  
1  English  
2  Spanish  
3  English  
4  Spanish  

TA

In [5]:
for i, table in enumerate(tables):
    if "Title" in table.columns:
        print(f"Table {i}")
        print(table.columns.tolist())
        print()

Table 0
['Title', 'Genre(s)', 'Premiere', 'Language']

Table 1
['Title', 'Premiere', 'Language']

Table 2
['Title', 'Premiere', 'Language']

Table 3
['Title', 'Premiere', 'Language']

Table 4
['Title', 'Genre', 'Release', 'Language']

Table 5
['Title', 'Subject', 'Premiere', 'Language']

Table 6
['Title', 'Genre']



In [6]:
df = []

for table in tables:

    table.columns = [
        col[0] if isinstance(col, tuple) else col
        for col in table.columns
    ]

    if "Title" in table.columns:

        # Some tables use Release instead of Premiere
        if "Release" in table.columns:
            table = table.rename(columns={"Release": "Premiere"})

        if "Premiere" in table.columns:
            df.append(table[["Title", "Premiere"]])

amazon_df = pd.concat(df, ignore_index=True)

amazon_df.drop_duplicates(subset=["Title"], inplace=True)

print(amazon_df.head())
print(amazon_df.shape)

                      Title            Premiere
0             Brad's Status  September 15, 2017
1                 Pass Over      April 20, 2018
2                  Peterloo       April 5, 2019
3           Guava Island[1]      April 13, 2019
4  Brittany Runs a Marathon   November 22, 2019
(471, 2)


In [7]:
amazon_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 471 entries, 0 to 470
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Title     471 non-null    str  
 1   Premiere  471 non-null    str  
dtypes: str(2)
memory usage: 7.5 KB


In [8]:
print(amazon_df["Title"].duplicated().sum())

0


In [9]:
amazon_df.sample(n=10, random_state=42)

,Title,Premiere
55,Coolie No. 1,"December 25, 2020"
73,Cold Case,"June 30, 2021"
33,Nocturne,"October 13, 2020"
446,Gina Brillon: The Floor Is Lava,"June 5, 2020"
425,Chris Ramsey: Approval Needed,"August 19, 2019"
229,Trunk – Locked in,"January 26, 2024"
210,The Day of the Dead Is Cancelled,"October 27, 2023"
9,Guns Akimbo,"March 23, 2020"
352,The Love Hypothesis,"September 23, 2026"
70,Ek Mini Katha,"May 27, 2021"


In [10]:
amazon_df['Title'] = amazon_df['Title'].str.replace(r'\[\d+\]', '', regex=True).str.strip()
amazon_df['Premiere'] = amazon_df['Premiere'].str.replace(r'\[\d+\]', '', regex=True).str.strip()

In [11]:
amazon_df["Title"] = (amazon_df["Title"].str.replace(r"\[[a-z]\]", "", regex=True).str.strip())
amazon_df[amazon_df["Title"].str.contains(r"\[", regex=True, na=False)]

,Title,Premiere


In [12]:
amazon_df.sample(n=10, random_state=42)

,Title,Premiere
55,Coolie No. 1,"December 25, 2020"
73,Cold Case,"June 30, 2021"
33,Nocturne,"October 13, 2020"
446,Gina Brillon: The Floor Is Lava,"June 5, 2020"
425,Chris Ramsey: Approval Needed,"August 19, 2019"
229,Trunk – Locked in,"January 26, 2024"
210,The Day of the Dead Is Cancelled,"October 27, 2023"
9,Guns Akimbo,"March 23, 2020"
352,The Love Hypothesis,"September 23, 2026"
70,Ek Mini Katha,"May 27, 2021"


In [13]:
amazon_df['Premiere'] = pd.to_datetime(amazon_df['Premiere'],errors='coerce')

In [14]:
cutoff_date = pd.Timestamp('2026-06-18')

In [15]:
premiered_df = amazon_df[(amazon_df['Premiere'].notna()) & (amazon_df['Premiere'] < cutoff_date)].copy()

premiered_df.to_csv('../data/movies/raw/amazon_titles.csv', index=False)

In [16]:
upcoming_tba_df = amazon_df[(amazon_df['Premiere'].isna()) | (amazon_df['Premiere'] >= cutoff_date)].copy()

upcoming_tba_df.to_csv('../data/movies/raw/upcoming_amazon.csv', index=False)

In [17]:
print(f"Premiered titles: {len(premiered_df)}")
print(f"Upcoming/TBA titles: {len(upcoming_tba_df)}")

Premiered titles: 462
Upcoming/TBA titles: 9


## Retrieving IMDb Metadata

In [ ]:
import requests

API_KEY = "YOUR_API_KEY"

title = "The Idea of You"

url = f"https://www.omdbapi.com/?t={title}&apikey={API_KEY}"

response = requests.get(url)

data = response.json()

print(data)

{'Title': 'The Idea of You', 'Year': '2024', 'Rated': 'R', 'Released': '02 May 2024', 'Runtime': '115 min', 'Genre': 'Comedy, Drama, Music', 'Director': 'Michael Showalter', 'Writer': 'Robinne Lee, Michael Showalter, Jennifer Westfeldt', 'Actors': 'Anne Hathaway, Nicholas Galitzine, Ella Rubin', 'Plot': 'Solène, a 40-year-old single mom, begins an unexpected romance with 24-year-old Hayes Campbell, the lead singer of August Moon, the hottest boy band on the planet.', 'Language': 'English', 'Country': 'United States', 'Awards': '2 wins & 11 nominations total', 'Poster': 'https://m.media-amazon.com/images/M/MV5BY2RkZmZjYTQtOGExMS00YTljLWJmNzQtMzI0ZDU3YzFjNzJkXkEyXkFqcGc@._V1_SX300.jpg', 'Ratings': [{'Source': 'Internet Movie Database', 'Value': '6.3/10'}, {'Source': 'Rotten Tomatoes', 'Value': '80%'}, {'Source': 'Metacritic', 'Value': '67/100'}], 'Metascore': '67', 'imdbRating': '6.3', 'imdbVotes': '82,785', 'imdbID': 'tt9466114', 'Type': 'movie', 'DVD': 'N/A', 'BoxOffice': 'N/A', 'Produ

In [20]:
def get_omdb_data(title):

    try:
        url = f"https://www.omdbapi.com/?t={title}&apikey={API_KEY}"

        data = requests.get(url).json()

        return pd.Series({
            "Rated": data.get("Rated"),
            "imdb_rating": data.get("imdbRating"),
            "imdb_votes": data.get("imdbVotes"),
            "imdb_genre": data.get("Genre"),
            "imdb_year": data.get("Year"),
            "imdb_type": data.get("Type")
        })

    except:
        return pd.Series()

In [21]:
amazon_df = pd.read_csv('../data/movies/raw/amazon_titles.csv')

In [22]:
omdb_data = amazon_df["Title"].apply(get_omdb_data)

amazon_df = pd.concat(
    [amazon_df, omdb_data],
    axis=1
)

In [23]:
amazon_df.head()

,Title,Premiere,Rated,imdb_rating,imdb_votes,imdb_genre,imdb_year,imdb_type
0,Brad's Status,2017-09-15,R,6.5,"19,482","Comedy, Drama, Music",2017,movie
1,Pass Over,2018-04-20,Not Rated,6.1,"1,150",Drama,2018,movie
2,Peterloo,2019-04-05,PG-13,6.5,"5,680","Drama, History",2018,movie
3,Guava Island,2019-04-13,TV-MA,6.6,"11,900","Comedy, Drama, Music",2019,movie
4,Brittany Runs a Marathon,2019-11-22,R,6.8,"23,753","Comedy, Drama",2019,movie


In [24]:
amazon_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 462 entries, 0 to 461
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Title        462 non-null    str  
 1   Premiere     462 non-null    str  
 2   Rated        420 non-null    str  
 3   imdb_rating  420 non-null    str  
 4   imdb_votes   420 non-null    str  
 5   imdb_genre   420 non-null    str  
 6   imdb_year    420 non-null    str  
 7   imdb_type    420 non-null    str  
dtypes: str(8)
memory usage: 29.0 KB


In [25]:
amazon_df['imdb_votes'] = (amazon_df['imdb_votes'].astype(str).str.replace(',', '', regex=False))

In [26]:
amazon_df['imdb_rating'] = pd.to_numeric(amazon_df['imdb_rating'], errors='coerce')
amazon_df['imdb_votes'] = pd.to_numeric(amazon_df['imdb_votes'], errors='coerce')

In [27]:
amazon_df['imdb_year'] = (amazon_df['imdb_year'].astype(str).str.extract(r'(\d{4})')[0]).astype('Int64')

amazon_df['imdb_year'] = pd.to_numeric(amazon_df['imdb_year'], errors='coerce')

In [28]:
amazon_df['imdb_rating'].isna().sum()

np.int64(88)

In [29]:
amazon_df = amazon_df.dropna(subset=['imdb_rating'])

In [30]:
amazon_df.info()

<class 'pandas.DataFrame'>
Index: 374 entries, 0 to 461
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        374 non-null    str    
 1   Premiere     374 non-null    str    
 2   Rated        374 non-null    str    
 3   imdb_rating  374 non-null    float64
 4   imdb_votes   374 non-null    float64
 5   imdb_genre   374 non-null    str    
 6   imdb_year    374 non-null    Int64  
 7   imdb_type    374 non-null    str    
dtypes: Int64(1), float64(2), str(5)
memory usage: 26.7 KB


In [31]:
amazon_df.to_csv("../data/movies/processed/amazon_with_imdb.csv",index=False)

In [32]:
amazon_df = pd.read_csv("../data/movies/processed/amazon_with_imdb.csv")

In [33]:
mismatch_df = amazon_df[pd.to_datetime(amazon_df['Premiere']).dt.year != amazon_df['imdb_year']]

print(len(mismatch_df))

79


In [34]:
mismatch_df['year_diff'] = (pd.to_datetime(mismatch_df['Premiere']).dt.year - mismatch_df['imdb_year']).abs()

print(mismatch_df['year_diff'].value_counts().sort_index())

year_diff
1     27
2      8
3      6
4      2
5      2
6      3
7      2
8      1
9      1
10     2
11     3
13     2
15     3
16     3
17     2
18     3
19     1
20     2
22     1
23     1
25     1
39     1
63     1
64     1
Name: count, dtype: int64


In [35]:
mismatch_df = mismatch_df.sort_values('year_diff', ascending=False)

mismatch_df.to_csv('../data/movies/processed/amazon_year_mismatches.csv', index=False)

print(f"Saved {len(mismatch_df)} mismatches")

Saved 79 mismatches


In [36]:
mismatch_df = pd.read_csv('../data/movies/processed/corrected_amazon_mismatches.csv')

print("Remove:", (mismatch_df['keep'].str.lower() == 'n').sum())
print("Update:", (mismatch_df['keep'].str.lower() == 'y').sum())

Remove: 67
Update: 12


In [37]:
print(amazon_df.columns.tolist())

['Title', 'Premiere', 'Rated', 'imdb_rating', 'imdb_votes', 'imdb_genre', 'imdb_year', 'imdb_type']


In [38]:
titles_to_remove = mismatch_df.loc[mismatch_df['keep'].str.lower() == 'n','Title']

amazon_df = amazon_df[~amazon_df['Title'].isin(titles_to_remove)].copy()

In [39]:
updates = mismatch_df[mismatch_df['keep'].str.lower() == 'y'].set_index('Title')
print(updates.columns.tolist())

['Premiere', 'imdb_rating', 'imdb_votes', 'imdb_genre', 'imdb_year', 'imdb_type', 'year_diff', 'keep']


In [40]:
amazon_df = amazon_df.set_index('Title')

cols_to_update = [
    'imdb_rating',
    'imdb_votes',
    'imdb_genre',
    'imdb_year',
    'imdb_type'
]

for col in cols_to_update:
    amazon_df.loc[updates.index, col] = updates[col]

amazon_df = amazon_df.reset_index()

In [41]:
print("Final rows:", len(amazon_df))

Final rows: 310


In [42]:
amazon_df['Title'].duplicated().sum()

np.int64(0)

In [43]:
amazon_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 310 entries, 0 to 309
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        310 non-null    str    
 1   Premiere     310 non-null    str    
 2   Rated        162 non-null    str    
 3   imdb_rating  310 non-null    float64
 4   imdb_votes   310 non-null    float64
 5   imdb_genre   310 non-null    str    
 6   imdb_year    310 non-null    int64  
 7   imdb_type    310 non-null    str    
dtypes: float64(2), int64(1), str(5)
memory usage: 19.5 KB


In [44]:
amazon_df.shape

(310, 8)

In [45]:
amazon_df.to_csv("../data/movies/processed/amazon_imdb.csv",index=False)

## Adaptation Verification and Saving the Dataset

In [46]:
import pandas as pd

df = pd.read_csv('../data/movies/processed/amazon_imdb_adaptations.csv')

df.head()

,Title,Premiere,Rated,imdb_rating,imdb_votes,imdb_genre,imdb_year,imdb_type,is_book_adaptation,book_title,book_author
0,Brad's Status,15/09/2017,R,6.5,19482,"Comedy, Drama, Music",2017,movie,n,NaN,NaN
1,Pass Over,20/04/2018,Not Rated,6.1,1150,Drama,2018,movie,n,NaN,NaN
2,Guava Island,13/04/2019,TV-MA,6.6,11900,"Comedy, Drama, Music",2019,movie,n,NaN,NaN
3,Brittany Runs a Marathon,22/11/2019,R,6.8,23753,"Comedy, Drama",2019,movie,n,NaN,NaN
4,The Report,29/11/2019,R,7.2,54874,"Biography, Crime, Drama",2019,movie,n,NaN,NaN


In [47]:
df.columns

Index(['Title', 'Premiere', 'Rated', 'imdb_rating', 'imdb_votes', 'imdb_genre',
       'imdb_year', 'imdb_type', 'is_book_adaptation', 'book_title',
       'book_author'],
      dtype='str')

In [48]:
df["is_book_adaptation"].value_counts()

is_book_adaptation
n    262
y     48
Name: count, dtype: int64

In [49]:
adaptations_df = df[df["is_book_adaptation"] == "y"].copy()

In [50]:
adaptations_df.shape

(48, 11)

In [51]:
adaptations_df[["book_title", "book_author"]].isnull().sum()

book_title     0
book_author    0
dtype: int64

In [52]:
adaptations_df["book_title"] = adaptations_df["book_title"].str.strip().str.lower()
adaptations_df["book_author"] = adaptations_df["book_author"].str.strip().str.lower()

In [53]:
adaptations_df[["book_title", "book_author"]].head(10)

,book_title,book_author
5,falling upwards: how we took to the air,richard holmes
6,bloodshot,"kevin vanhook, don perlin, bob layton"
12,radioactive: marie & pierre curie: a tale of l...,lauren redniss
15,our chemical hearts,krystal sutherland
18,the postcard killers,james patterson & liza marklund
20,evil eye,madhuri shekar
26,simply fly: a deccan odyssey,g. r. gopinath
42,the map of tiny perfect things,lev grossman
47,without remorse,tom clancy
62,le bal des folles (the mad women's ball),victoria mas


In [54]:
import re

def clean_title(text):
    text = text.lower()
    text = re.sub(r"\(.*?\)", "", text)   # remove brackets content
    text = text.replace(",", " ")
    text = re.sub(r"\s+", " ", text)     # remove extra spaces
    return text.strip()

adaptations_df["book_title"] = adaptations_df["book_title"].apply(clean_title)

In [55]:
def to_search_key(text):
    text = text.lower()
    text = re.sub(r"(series|universe|novels|books)", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

adaptations_df["search_key"] = adaptations_df["book_title"].apply(to_search_key)

In [56]:
adaptations_df.head(10)

,Title,Premiere,Rated,imdb_rating,imdb_votes,imdb_genre,imdb_year,imdb_type,is_book_adaptation,book_title,book_author,search_key
5,The Aeronauts,20/12/2019,PG-13,6.6,39885,"Action, Adventure, Drama",2019,movie,y,falling upwards: how we took to the air,richard holmes,falling upwards: how we took to the air
6,Bloodshot,27/03/2020,PG-13,5.7,90285,"Action, Adventure, Sci-Fi",2020,movie,y,bloodshot,"kevin vanhook, don perlin, bob layton",bloodshot
12,Radioactive,24/07/2020,PG-13,6.3,23792,"Biography, Drama, Romance",2019,movie,y,radioactive: marie & pierre curie: a tale of l...,lauren redniss,radioactive: marie & pierre curie: a tale of l...
15,Chemical Hearts,21/08/2020,R,6.3,17270,"Drama, Romance",2020,movie,y,our chemical hearts,krystal sutherland,our chemical hearts
18,The Postcard Killings,25/09/2020,Not Rated,5.8,18316,"Crime, Drama, Mystery",2020,movie,y,the postcard killers,james patterson & liza marklund,the postcard killers
20,Evil Eye,13/10/2020,Not Rated,4.8,4377,"Horror, Mystery, Thriller",2020,movie,y,evil eye,madhuri shekar,evil eye
26,Soorarai Pottru,12/11/2020,TV-MA,8.7,127179,Drama,2020,movie,y,simply fly: a deccan odyssey,g. r. gopinath,simply fly: a deccan odyssey
42,The Map of Tiny Perfect Things,12/02/2021,PG-13,6.8,32443,"Comedy, Fantasy, Romance",2021,movie,y,the map of tiny perfect things,lev grossman,the map of tiny perfect things
47,Without Remorse,30/04/2021,R,5.8,70410,"Action, Drama, Thriller",2021,movie,y,without remorse,tom clancy,without remorse
62,The Mad Women's Ball,17/09/2021,NaN,6.7,4727,"Drama, Thriller",2021,movie,y,le bal des folles,victoria mas,le bal des folles


In [57]:
adaptations_df.duplicated().sum()

np.int64(0)

In [58]:
adaptations_df.shape

(48, 12)

In [59]:
adaptations_df.to_csv("../data/movies/final/amazon_movies.csv", index=False)

# Netflix Movies Data Collection

## Web Scraping Wikipedia Data

In [60]:
import pandas as pd
import requests
from io import StringIO

urls = [
    "https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2015%E2%80%932017)",
    "https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2018)",
    "https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2019)",
    "https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2020)",
    "https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2021)",
    "https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2022)",
    "https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2023)",
    "https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2024)",
    "https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2025)",
    "https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(since_2026)"
]

headers = {
    "User-Agent": "Mozilla/5.0"
}

In [61]:
all_tables = []

for url in urls:
    print(f"Reading: {url}")

    html = requests.get(url, headers=headers).text

    tables = pd.read_html(StringIO(html))

    for table in tables:

        # Flatten multi-level columns if present
        table.columns = [
            col[0] if isinstance(col, tuple) else col
            for col in table.columns
        ]

        cols = [str(c).strip() for c in table.columns]

        if "Title" in cols:

            table.columns = cols

            # Different pages may use different date column names
            date_col = None

            for candidate in [
                "Premiere",
                "Release date",
                "Release Date",
                "Date"
            ]:
                if candidate in table.columns:
                    date_col = candidate
                    break

            if date_col:
                temp = table[["Title", date_col]].copy()
                temp.rename(columns={date_col: "Premiere"}, inplace=True)
                all_tables.append(temp)

netflix_df = pd.concat(all_tables, ignore_index=True)

netflix_df.drop_duplicates(subset=["Title"], inplace=True)

netflix_df.reset_index(drop=True, inplace=True)

netflix_df.shape

Reading: https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2015%E2%80%932017)
Reading: https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2018)
Reading: https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2019)
Reading: https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2020)
Reading: https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2021)
Reading: https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2022)
Reading: https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2023)
Reading: https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2024)
Reading: https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(2025)
Reading: https://en.wikipedia.org/wiki/List_of_Netflix_original_films_(since_2026)


(1803, 2)

In [62]:
netflix_df.head()

,Title,Premiere
0,Beasts of No Nation,"October 16, 2015"
1,The Ridiculous 6,"December 11, 2015"
2,"Crouching Tiger, Hidden Dragon: Sword of Destiny","February 26, 2016"
3,Pee-wee's Big Holiday,"March 18, 2016"
4,Special Correspondents,"April 29, 2016"


In [63]:
netflix_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1803 entries, 0 to 1802
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Title     1803 non-null   str  
 1   Premiere  1803 non-null   str  
dtypes: str(2)
memory usage: 28.3 KB


In [64]:
netflix_df = netflix_df.drop_duplicates()

In [65]:
netflix_df.shape

(1803, 2)

In [66]:
netflix_df.sample(n=10, random_state=42)

,Title,Premiere
832,Drifting Home,"September 16, 2022"
613,Friendzone,"September 29, 2021"
1582,Selena y Los Dinos: A Family's Legacy,"November 17, 2025"
162,The Night Comes for Us,"October 19, 2018"
596,The Kissing Booth 3,"August 11, 2021"
70,I'll Sleep When I'm Dead,"August 19, 2016"
637,We Couldn't Become Adults,"November 5, 2021"
220,Triple Frontier,"March 13, 2019"
1137,Lewis Capaldi: How I'm Feeling Now,"April 5, 2023"
1007,Kill Boksoon,"March 31, 2023"


In [67]:
netflix_df['Title'] = netflix_df['Title'].str.replace(r'\[\d+\]', '', regex=True).str.strip()
netflix_df['Premiere'] = netflix_df['Premiere'].str.replace(r'\[\d+\]', '', regex=True).str.strip()

In [68]:
netflix_df["Title"] = (netflix_df["Title"].str.replace(r"\[[a-z]\]", "", regex=True).str.strip())

In [69]:
netflix_df[netflix_df["Title"].str.contains(r"\[", regex=True, na=False)]

,Title,Premiere


In [70]:
netflix_df.sample(n=10, random_state=42)

,Title,Premiere
832,Drifting Home,"September 16, 2022"
613,Friendzone,"September 29, 2021"
1582,Selena y Los Dinos: A Family's Legacy,"November 17, 2025"
162,The Night Comes for Us,"October 19, 2018"
596,The Kissing Booth 3,"August 11, 2021"
70,I'll Sleep When I'm Dead,"August 19, 2016"
637,We Couldn't Become Adults,"November 5, 2021"
220,Triple Frontier,"March 13, 2019"
1137,Lewis Capaldi: How I'm Feeling Now,"April 5, 2023"
1007,Kill Boksoon,"March 31, 2023"


In [71]:
netflix_df['Premiere'] = pd.to_datetime(netflix_df['Premiere'],errors='coerce')

In [72]:
cutoff_date = pd.Timestamp('2026-06-18')

In [73]:
premiered_df = netflix_df[(netflix_df['Premiere'] .notna()) & (netflix_df['Premiere'] < cutoff_date)].copy()

upcoming_tba_df = netflix_df[(netflix_df['Premiere'].isna()) | (netflix_df['Premiere'] >= cutoff_date)].copy()

In [74]:
premiered_df.to_csv('../data/movies/raw/netflix_titles.csv', index=False)
upcoming_tba_df.to_csv('../data/movies/raw/upcoming_netflix.csv', index=False)

In [75]:
print(f"Premiered titles: {len(premiered_df)}")
print(f"Upcoming/TBA titles: {len(upcoming_tba_df)}")

Premiered titles: 1692
Upcoming/TBA titles: 111


## Adaptation Verification

In [76]:
import pandas as pd

df = pd.read_csv("../data/movies/processed/netflix_adaptations.csv")

df.head()

,Title,Premiere,is_book_adaptation,book_title,book_author
0,Beasts of No Nation,16/10/2015,y,Beasts of No Nation,Uzodinma Iweala
1,The Ridiculous 6,11/12/2015,n,NaN,NaN
2,"Crouching Tiger, Hidden Dragon: Sword of Destiny",26/02/2016,y,"Iron Knight, Silver Vase",Wang Dulu
3,Pee-wee's Big Holiday,18/03/2016,n,NaN,NaN
4,Special Correspondents,29/04/2016,n,NaN,NaN


In [77]:
df["is_book_adaptation"].value_counts()

is_book_adaptation
n    1511
y     181
Name: count, dtype: int64

In [78]:
adaptations_df = df[df["is_book_adaptation"] == "y"].copy()

In [79]:
adaptations_df.shape

(181, 5)

In [80]:
adaptations_df[["book_title", "book_author"]].isnull().sum()

book_title     0
book_author    0
dtype: int64

In [81]:
adaptations_df["book_title"] = adaptations_df["book_title"].str.strip().str.lower()
adaptations_df["book_author"] = adaptations_df["book_author"].str.strip().str.lower()

In [82]:
adaptations_df[["book_title", "book_author"]].head(10)

,book_title,book_author
0,beasts of no nation,uzodinma iweala
2,"iron knight, silver vase",wang dulu
6,the revised fundamentals of caregiving,jonathan evison
20,coin heist,elisa ludwig
23,iboy,kevin brooks
37,blame!,tsutomu nihei
44,death note,tsugumi ohba & takeshi obata
47,first they killed my father: a daughter of cam...,loung ung
48,gerald's game,stephen king
49,our souls at night,kent haruf


In [83]:
import re

def clean_title(text):
    text = text.lower()
    text = re.sub(r"\(.*?\)", "", text)   # remove brackets content
    text = text.replace(",", " ")
    text = re.sub(r"\s+", " ", text)     # remove extra spaces
    return text.strip()

adaptations_df["book_title"] = adaptations_df["book_title"].apply(clean_title)

In [84]:
def to_search_key(text):
    text = text.lower()
    text = re.sub(r"(series|universe|novels|books)", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

adaptations_df["search_key"] = adaptations_df["book_title"].apply(to_search_key)

In [85]:
adaptations_df.head(10)

,Title,Premiere,is_book_adaptation,book_title,book_author,search_key
0,Beasts of No Nation,16/10/2015,y,beasts of no nation,uzodinma iweala,beasts of no nation
2,"Crouching Tiger, Hidden Dragon: Sword of Destiny",26/02/2016,y,iron knight silver vase,wang dulu,iron knight silver vase
6,The Fundamentals of Caring,24/06/2016,y,the revised fundamentals of caregiving,jonathan evison,the revised fundamentals of caregiving
20,Coin Heist,06/01/2017,y,coin heist,elisa ludwig,coin heist
23,iBOY,27/01/2017,y,iboy,kevin brooks,iboy
37,BLAME!,20/05/2017,y,blame!,tsutomu nihei,blame!
44,Death Note,25/08/2017,y,death note,tsugumi ohba & takeshi obata,death note
47,First They Killed My Father,15/09/2017,y,first they killed my father: a daughter of cam...,loung ung,first they killed my father: a daughter of cam...
48,Gerald's Game,29/09/2017,y,gerald's game,stephen king,gerald's game
49,Our Souls at Night,29/09/2017,y,our souls at night,kent haruf,our souls at night


In [86]:
adaptations_df.duplicated().sum()

np.int64(0)

In [87]:
adaptations_df.to_csv("../data/movies/processed/netflix_adaptations_clean.csv", index=False)

## Retrieving IMDb Metadata and Saving the Dataset

In [ ]:
import pandas as pd
import requests

API_KEY = "YOUR_API_KEY"

def get_omdb_data(title):
    try:
        url = f"https://www.omdbapi.com/?t={title}&apikey={API_KEY}"

        data = requests.get(url).json()

        return pd.Series({
            "Rated": data.get("Rated"),
            "imdb_rating": data.get("imdbRating"),
            "imdb_votes": data.get("imdbVotes"),
            "imdb_genre": data.get("Genre"),
            "imdb_year": data.get("Year"),
            "imdb_type": data.get("Type")
        })

    except:
        return pd.Series()

In [89]:
netflix_df = pd.read_csv('../data/movies/processed/netflix_adaptations_clean.csv')

In [90]:
omdb_data = netflix_df["Title"].apply(get_omdb_data)

netflix_df = pd.concat([netflix_df, omdb_data], axis=1)

netflix_df.head()

,Title,Premiere,is_book_adaptation,book_title,book_author,search_key,Rated,imdb_rating,imdb_votes,imdb_genre,imdb_year,imdb_type
0,Beasts of No Nation,16/10/2015,y,beasts of no nation,uzodinma iweala,beasts of no nation,TV-MA,7.7,"90,320","Drama, War",2015,movie
1,"Crouching Tiger, Hidden Dragon: Sword of Destiny",26/02/2016,y,iron knight silver vase,wang dulu,iron knight silver vase,PG-13,6.1,"21,585","Action, Adventure, Drama",2016,movie
2,The Fundamentals of Caring,24/06/2016,y,the revised fundamentals of caregiving,jonathan evison,the revised fundamentals of caregiving,TV-MA,7.3,"81,560","Comedy, Drama",2016,movie
3,Coin Heist,06/01/2017,y,coin heist,elisa ludwig,coin heist,TV-14,4.8,"3,039","Crime, Drama, Romance",2017,movie
4,iBOY,27/01/2017,y,iboy,kevin brooks,iboy,TV-MA,5.9,"24,700","Action, Crime, Sci-Fi",2017,movie


In [91]:
netflix_df.shape

(181, 12)

In [92]:
netflix_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 181 entries, 0 to 180
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Title               181 non-null    str  
 1   Premiere            181 non-null    str  
 2   is_book_adaptation  181 non-null    str  
 3   book_title          181 non-null    str  
 4   book_author         181 non-null    str  
 5   search_key          181 non-null    str  
 6   Rated               171 non-null    str  
 7   imdb_rating         171 non-null    str  
 8   imdb_votes          171 non-null    str  
 9   imdb_genre          171 non-null    str  
 10  imdb_year           171 non-null    str  
 11  imdb_type           171 non-null    str  
dtypes: str(12)
memory usage: 17.1 KB


In [93]:
netflix_df['imdb_votes'] = (netflix_df['imdb_votes'].astype(str).str.replace(',', '', regex=False))

In [94]:
netflix_df['imdb_rating'] = pd.to_numeric(netflix_df['imdb_rating'], errors='coerce')
netflix_df['imdb_votes'] = pd.to_numeric(netflix_df['imdb_votes'], errors='coerce')

In [95]:
netflix_df['imdb_year'] = (netflix_df['imdb_year'].astype(str).str.extract(r'(\d{4})')[0].astype('Int64'))

In [96]:
netflix_df['imdb_year'] = pd.to_numeric(netflix_df['imdb_year'], errors='coerce')

In [97]:
netflix_df = netflix_df.dropna(subset=['imdb_rating'])

In [98]:
netflix_df.info()

<class 'pandas.DataFrame'>
Index: 163 entries, 0 to 179
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               163 non-null    str    
 1   Premiere            163 non-null    str    
 2   is_book_adaptation  163 non-null    str    
 3   book_title          163 non-null    str    
 4   book_author         163 non-null    str    
 5   search_key          163 non-null    str    
 6   Rated               163 non-null    str    
 7   imdb_rating         163 non-null    float64
 8   imdb_votes          163 non-null    float64
 9   imdb_genre          163 non-null    str    
 10  imdb_year           163 non-null    Int64  
 11  imdb_type           163 non-null    str    
dtypes: Int64(1), float64(2), str(9)
memory usage: 16.7 KB


In [99]:
netflix_df = netflix_df[
    [
        "Title",
        "Premiere",
        "Rated",
        "imdb_rating",
        "imdb_votes",
        "imdb_genre",
        "imdb_year",
        "imdb_type",
        "is_book_adaptation",
        "book_title",
        "book_author",
        "search_key"
    ]
]

In [100]:
netflix_df.info()

<class 'pandas.DataFrame'>
Index: 163 entries, 0 to 179
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               163 non-null    str    
 1   Premiere            163 non-null    str    
 2   Rated               163 non-null    str    
 3   imdb_rating         163 non-null    float64
 4   imdb_votes          163 non-null    float64
 5   imdb_genre          163 non-null    str    
 6   imdb_year           163 non-null    Int64  
 7   imdb_type           163 non-null    str    
 8   is_book_adaptation  163 non-null    str    
 9   book_title          163 non-null    str    
 10  book_author         163 non-null    str    
 11  search_key          163 non-null    str    
dtypes: Int64(1), float64(2), str(9)
memory usage: 16.7 KB


In [101]:
mismatch_df = netflix_df[pd.to_datetime(netflix_df['Premiere'], dayfirst=True).dt.year != netflix_df['imdb_year']]

print(len(mismatch_df))

18


In [102]:
mismatch_df['year_diff'] = (pd.to_datetime(mismatch_df['Premiere'], dayfirst=True).dt.year - mismatch_df['imdb_year']).abs()

print(mismatch_df['year_diff'].value_counts().sort_index())

year_diff
1     5
2     1
4     1
5     1
7     2
11    1
17    1
18    1
24    1
31    1
34    1
71    1
80    1
Name: count, dtype: Int64


In [103]:
mismatch_df = mismatch_df.sort_values('year_diff', ascending=False)

mismatch_df.to_csv('../data/movies/processed/netflix_year_mismatches.csv', index=False)

print(f"Saved {len(mismatch_df)} mismatches")

Saved 18 mismatches


In [104]:
mismatch_df = pd.read_csv('../data/movies/processed/corrected_netflix_mismatches.csv')

print("Remove:", (mismatch_df['keep'].str.lower() == 'n').sum())
print("Update:", (mismatch_df['keep'].str.lower() == 'y').sum())

Remove: 1
Update: 17


In [105]:
titles_to_remove = mismatch_df.loc[mismatch_df['keep'].str.lower() == 'n','Title']

netflix_df = netflix_df[~netflix_df['Title'].isin(titles_to_remove)].copy()

In [106]:
updates = mismatch_df[mismatch_df['keep'].str.lower() == 'y'].set_index('Title')

netflix_df = netflix_df.set_index('Title')

cols_to_update = [
    'Rated',
    'imdb_rating',
    'imdb_votes',
    'imdb_genre',
    'imdb_year',
    'imdb_type'
]

for col in cols_to_update:
    netflix_df.loc[updates.index, col] = updates[col]

netflix_df = netflix_df.reset_index()

In [108]:
print("Final rows:", len(netflix_df))

Final rows: 162


In [109]:
netflix_df.to_csv("../data/movies/final/netflix_movies.csv",index=False)